## 1. Формулювання перевірюваних гіпотез

Перед запуском експериментів фіксуємо три ключові гіпотези:

### Гіпотеза 1: Вплив складності на Train та CV похибки
* **Гіпотеза:** Збільшення степеня полінома (складності моделі) призведе до монотонного зменшення помилки на навчальній вибірці ($Train\ RMSE$), але помилка на валідації ($CV\ RMSE$) матиме U-подібну форму (спочатку зменшиться, а потім почне зростати).
* **Що обчислюється:** Середні значення $Train\ RMSE$ та $CV\ RMSE$ для лінійної регресії при різних степенях полінома ($d \in [1, 2, 3, 4]$).
* **Графік/Таблиця:** Графік валідаційної кривої (залежність Train та CV похибок від степеня полінома).
* **Критерій підтвердження:** Train-крива постійно спадає, CV-крива досягає мінімуму (наприклад, при $d=2$), після чого різко йде вгору.
* **Критерій спростування:** CV-крива продовжує спадати або не має вираженого зростання зі збільшенням $d$.

### Гіпотеза 2: Поява перенавчання (Overfitting)
* **Гіпотеза:** Моделі з високим степенем полінома ($d \ge 3$) без регуляризації будуть сильно перенавчатися, що проявиться у великому розриві узагальнення.
* **Що обчислюється:** Розрив узагальнення (Generalization Gap = $CV\ RMSE - Train\ RMSE$).
* **Графік/Таблиця:** Таблиця порівняння $Train$, $CV$ похибок та їхньої різниці для кожного степеня полінома.
* **Критерій підтвердження:** Розрив узагальнення для $d \ge 3$ стає в кілька разів більшим, ніж для $d=1$.
* **Критерій спростування:** Розрив узагальнення залишається сталим незалежно від степеня полінома.

### Гіпотеза 3: Вплив регуляризації
* **Гіпотеза:** Застосування Elastic Net до складної поліноміальної моделі ($d=3$) зменшить розрив узагальнення та дисперсію ($Std\ CV\ RMSE$), зануляючи неважливі ознаки (ефект $L_1$) та обмежуючи ваги (ефект $L_2$).
* **Що обчислюється:** $CV\ RMSE$, розрив узагальнення, $Std\ CV\ RMSE$, кількість ненульових коефіцієнтів моделі.
* **Графік/Таблиця:** Таблиця порівняння нерегуляризованої моделі ($d=3$) та найкращої Elastic Net ($d=3$).
* **Критерій підтвердження:** "Покращення" фіксується, якщо Elastic Net демонструє: 1) менший або рівний $CV\ RMSE$; 2) менший розрив узагальнення; 3) менший $Std\ RMSE$ (вищу стабільність); 4) нульові ваги для частини ознак (простіша модель).
* **Критерій спростування:** Регуляризована модель показує гірший $CV\ RMSE$ без суттєвого зменшення розриву узагальнення.

---

## 2. Фіксація експериментального протоколу

До початку виконання розрахунків зафіксовано такі параметри експерименту:

* **Вибірки:** Train (80%), Test (20%). Розбиття зафіксовано через `random_state=42`. Test set ізолюється.
* **Схема крос-валідації:** `KFold(n_splits=5, shuffle=True, random_state=42)`. Усі моделі оцінюються на цих ідентичних фолдах.
* **Основна метрика:** $RMSE$ (Root Mean Squared Error).
* **Структура Pipeline:**
  1. `ColumnTransformer` (StandardScaler для числових, OHE для категоріальних).
  2. `PolynomialFeatures` (для генерації поліномів).
  3. Модель (Linear Regression або ElasticNet).
* **Степені полінома:** $d \in [1, 2, 3, 4]$.
* **Сітка гіперпараметрів Elastic Net:**
  * `alpha`: `[0.001, 0.01, 0.1, 1.0, 10.0, 100.0]`
  * `l1_ratio`: `[0.0, 0.25, 0.5, 0.75, 1.0]` (де 0 - Ridge, 1 - Lasso).
* **Критерій вибору фінальної моделі:** Модель із найменшим середнім $CV\ RMSE$. За умови різниці $CV\ RMSE$ у межах $1\%$, обирається простіша модель (менший степінь $d$ або більша регуляризація).
* **Початковий бюджет пошуку:** 
  * Linear Regression: 4 конфігурації (за степенями полінома).
  * Elastic Net: $4 \times 6 \times 5 = 120$ конфігурацій.
  * Загалом: 124 конфігурації.
* **Протокол зміни плану:** Якщо найкраще значення гіперпараметра опиниться на межі сітки (наприклад, `alpha=0.001` або `alpha=100.0`), сітка буде розширена в цей бік, про що буде зроблено окремий запис із зазначенням попереднього найкращого результату.